# Train issue-relation reranker

Run this notebook with a **T4 GPU**. Upload `relation_reranker_colab_bundle.zip` when prompted. The data contains scrubbed but still internal Jira business text; use only an approved Colab account/runtime.

The run has a fixed family-level split and fixed hyperparameters. It saves weights only when every held-out gate passes. A pass is development evidence, not an automatic production deployment.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect.'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'relation_reranker_colab_bundle.zip' in uploaded
!rm -rf /content/relation_reranker_run
!mkdir -p /content/relation_reranker_run
!unzip -q relation_reranker_colab_bundle.zip -d /content/relation_reranker_run
%cd /content/relation_reranker_run

In [ ]:
!pip -q install 'transformers>=4.44,<5' sentencepiece safetensors
!python train_relation_reranker.py --data data --out output --epochs 1 --batch-size 8 --grad-accum 2 --max-length 256

In [ ]:
import json
from pathlib import Path
report = json.loads(Path('output/report.json').read_text(encoding='utf-8'))
print('ACCEPTED:', report['accepted'])
print(json.dumps(report['gates'], indent=2, ensure_ascii=False))
print(json.dumps(report['production_baseline_vs_trained'], indent=2, ensure_ascii=False))

In [ ]:
!zip -qr relation_reranker_result.zip output
from google.colab import files
files.download('relation_reranker_result.zip')